In [0]:
%python
dbutils.library.restartPython()

In [0]:
%python
import sys  # fixed: sys was never imported
sys.path.append("/Workspace/Users/alliesdl@gmail.com/credit_risk_pipeline")  # fixed: leading /

from pyspark.sql import functions as F, Window
from src.cleaning import *
from src.quality import assert_unique, unparsed_dates, orphans, null_report  # fixed: orphans

LINEAGE = ["_source_file", "_source_sheet", "_ingested_at"]

c = spark.table("lender_portfolio.bronze.customers").drop(*LINEAGE)  # fixed: portfolio

report_unmapped(c, "province", PROVINCE).show()
report_unmapped(c, "sex", SEX).show()  # fixed: SEX without quotes

c = clean_id(c)
c = c.withColumn("bureau_no_hit", F.col("bureau_score") == "NH")
c = parse_money(c, "monthly_income")
for col in ["employment_months", "bureau_score"]:
    c = parse_number(c, col)
c = parse_date(c, "signup_date")
c = map_values(c, "sex", SEX)
c = map_values(c, "province", PROVINCE, keep_unmapped=False)

c = null_outside(c, "monthly_income", 1000, 20000)
c = null_outside(c, "employment_months", 0, 600)
c = null_outside(c, "bureau_score", 300, 850)

yy = F.expr("try_cast(substring(id_number, 1, 2) as int)")  # fixed: expr
c = c.withColumn("birth_year", F.when(yy <= 10, 2000 + yy).otherwise(1900 + yy))
c = c.drop("id_number", "first_name", "surname", "cell")

before = c.count()
c = c.dropDuplicates()
print("exact duplicate rows removed:", before - c.count())

assert_unique(c, "customer_id")
assert unparsed_dates(c, "signup_date") == 0
print(c.count(), null_report(c))

(c.write.format("delta")
   .mode("overwrite")
   .option("overwriteSchema", "true")
   .saveAsTable("lender_portfolio.silver.customers"))

In [0]:
%python

# ---------- Applications ----------
a = spark.table("lender_portfolio.bronze.applications").drop(*LINEAGE)
a = clean_id(a)
a = to_int(a, "application_id")
a = parse_date(a, "application_date")
a = parse_money(a, "requested_amount")
for col in ["term_months", "interest_rate_pct"]:
    a = parse_number(a, col)  # fixed: indented
report_unmapped(a, "decision", DECISION).show()
a = map_values(a, "decision", DECISION, keep_unmapped=False)
a = a.withColumn("partner_id", F.coalesce("partner_id", F.lit("UNKNOWN")))

before = a.count()
a = a.dropDuplicates()
print("duplicate applications removed:", before - a.count())
assert_unique(a, "application_id")
assert unparsed_dates(a, "application_date") == 0
(a.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
   .saveAsTable("lender_portfolio.silver.applications"))

# ---------- Loans ----------
l = spark.table("lender_portfolio.bronze.loans").drop(*LINEAGE)
l = clean_id(l)
l = to_int(l, "loan_id", "application_id")
l = parse_date(l, "disbursement_date")
l = parse_money(l, "principal")
for col in ["term_months", "interest_rate_pct"]:
    l = parse_number(l, col)  # fixed: indented
l = l.dropDuplicates()
assert_unique(l, "loan_id")
assert unparsed_dates(l, "disbursement_date") == 0
(l.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
   .saveAsTable("lender_portfolio.silver.loans"))

# ---------- Partners ----------
p = spark.table("lender_portfolio.bronze.partners").drop(*LINEAGE)
p = parse_number(p, "commission_pct")
assert_unique(p, "partner_id")
(p.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
   .saveAsTable("lender_portfolio.silver.partners"))

print("applications", a.count(), "loans", l.count(), "partners", p.count())

In [0]:
%python

r = spark.table("lender_portfolio.bronze.repayments").drop(*LINEAGE)
r = to_int(r, "loan_id", "instalment_no")
r = parse_date(r, "due_date")
for col in ["amount_due", "amount_paid", "days_past_due"]:
    r = parse_number(r, col)  # fixed: indented
r = null_outside(r, "amount_paid", 0, 1e7)  # a negative payment is a capture error, not a refund

before = r.count()
r = r.dropDuplicates()
print("duplicate instalments removed:", before - r.count())
assert_unique(r, ["loan_id", "instalment_no"])
assert unparsed_dates(r, "due_date") == 0
(r.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
   .saveAsTable("lender_portfolio.silver.repayments"))
print("repayments", r.count())

# ---------- Referential checks: every child row should have a parent ----------
cu = spark.table("lender_portfolio.silver.customers")
ap = spark.table("lender_portfolio.silver.applications")
lo = spark.table("lender_portfolio.silver.loans")
print("applications without a customer:", orphans(ap, cu, "customer_id"))
print("loans without an application:  ", orphans(lo, ap, "application_id"))
print("repayments without a loan:     ", orphans(r, lo, "loan_id"))

In [0]:
%python

c = spark.table("lender_portfolio.bronze.campaigns").drop(*LINEAGE)

c = clean_id(c, "partner_id", prefix="P", width=2) 
c = c.withColumn("campaign_name", F.initcap(F.lower(F.trim("campaign_name"))))
c = parse_date(c, "start_date")
c = parse_date(c, "end_date")
c = parse_money(c, "spend_zar") 

c = (c.withColumn("dates_swapped", F.col("start_date") > F.col("end_date"))
      .withColumn("s", F.least("start_date", "end_date"))
      .withColumn("e", F.greatest("start_date", "end_date"))
      .drop("start_date", "end_date")
      .withColumnRenamed("s", "start_date").withColumnRenamed("e", "end_date"))

c = (c.withColumn("spend_is_credit", F.col("spend_zar") < 0)
      .withColumn("spend_zar", F.when(F.col("spend_zar") >= 0, F.col("spend_zar"))))

before = c.count()
c = c.dropDuplicates()
print("exact duplicate rows removed:", before - c.count())

assert_unique(c, "campaign_id")
print("start dates not parsed:", unparsed_dates(c, "start_date"), "| end dates not parsed:", unparsed_dates(c, "end_date"))
parts = spark.table("lender_portfolio.silver.partners")
print("campaigns with an unknown partner:", orphans(c, parts, "partner_id"))
print("swapped dates fixed:", c.filter("dates_swapped").count(), "| credit notes:", c.filter("spend_is_credit").count(),
      "| spend missing:", c.filter("spend_zar IS NULL").count())

(c.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
   .saveAsTable("lender_portfolio.silver.campaigns"))
print("silver.campaigns", c.count())